### Erase Database and Restart

To ensure a clean start, you can delete the existing `my_database.db` file before creating new tables. This will remove all data and table structures, allowing you to re-run the table creation and data insertion steps from scratch.

In [ ]:
import os

db_file = 'my_database.db'
# Close the connection if it's open before attempting to delete the file
# This check assumes 'conn' is a global variable or accessible in this scope.
# If you're running this after an error, 'conn' might not be defined or open.
# A more robust solution might involve wrapping this in a function or checking 'conn' status.
try:
    if 'conn' in locals() and conn:
        conn.close()
except NameError:
    pass # 'conn' was not defined

if os.path.exists(db_file):
    os.remove(db_file)
    print(f'Removed existing database file: {db_file}')
else:
    print(f'Database file {db_file} does not exist. No need to remove.')

Removed existing database file: my_database.db


In [ ]:
import sqlite3

# Create (or connect to) a database file
conn = sqlite3.connect('my_database.db')

# Create a cursor
cursor = conn.cursor()

In [ ]:
# Section A: Create Tables
cursor.execute("""
CREATE TABLE IF NOT EXISTS Outlet (
    OutletID INTEGER PRIMARY KEY,
    Address TEXT,
    PhoneNumber INTEGER,
    FaxNumber INTEGER
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS Vehicle (
    RegistrationNumber TEXT PRIMARY KEY,
    OutletID INTEGER NOT NULL,
    Model TEXT,
    Make TEXT,
    EngineSize REAL,
    Capacity REAL,
    CurrentMileage INTEGER,
    DailyHireRate REAL,
    CurrentLocation TEXT,
    CHECK (CurrentMileage >= 0),
    CHECK (DailyHireRate > 0),
    CHECK (EngineSize > 0),
    CHECK (Capacity > 0),
    FOREIGN KEY (OutletID) REFERENCES Outlet(OutletID)
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS Client (
    ClientID INTEGER PRIMARY KEY,
    FirstName TEXT,
    LastName TEXT,
    Address TEXT,
    PhoneNumber INTEGER,
    DateOfBirth TEXT,
    DrivingLicenseNumber TEXT UNIQUE
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS Staff (
    StaffID INTEGER PRIMARY KEY,
    OutletID INTEGER NOT NULL,
    FirstName TEXT,
    LastName TEXT,
    Address TEXT,
    PhoneNumber INTEGER,
    DateOfBirth TEXT,
    Sex TEXT CHECK (Sex IN ('M', 'F', 'Other')),
    DateJoined TEXT,
    JobTitle TEXT,
    Salary REAL,
    CHECK (Salary > 0),
    FOREIGN KEY (OutletID) REFERENCES Outlet(OutletID)
)
""")
cursor.execute("""
CREATE TABLE IF NOT EXISTS HireAgreement (
    HireID INTEGER PRIMARY KEY,
    RegistrationNumber TEXT NOT NULL,
    ClientID INTEGER NOT NULL,
    StartDate TEXT,
    EndDate TEXT,
    PreMileage REAL,
    PostMileage REAL,
    CHECK (EndDate >= StartDate),
    CHECK (PreMileage >= 0),
    CHECK (PostMileage >= 0),
    CHECK (PostMileage >= PreMileage),
    FOREIGN KEY (RegistrationNumber) REFERENCES Vehicle(RegistrationNumber),
    FOREIGN KEY (ClientID) REFERENCES Client(ClientID)
)
""")

In [ ]:
# Enforce Age Requiremnt
# Must be of Legal Driving Age
cursor.execute("""
CREATE TRIGGER check_client_age
BEFORE INSERT ON HireAgreement
FOR EACH ROW
BEGIN
    SELECT RAISE(ABORT, 'Client does not meet the minimum driving age of 16')
    WHERE (
        SELECT (strftime('%Y', 'now') - strftime('%Y', DateOfBirth)
              - (strftime('%m-%d', 'now') < strftime('%m-%d', DateOfBirth)))
        FROM Client WHERE ClientID = NEW.ClientID
    ) < 16;
END;
""")

# No overlapping hire dates
cursor.execute("""
CREATE TRIGGER check_no_overlapping_hires
BEFORE INSERT ON HireAgreement
FOR EACH ROW
BEGIN
    SELECT RAISE(ABORT, 'Vehicle already has an active hire agreement during this period')
    WHERE EXISTS (
        SELECT 1 FROM HireAgreement
        WHERE RegistrationNumber = NEW.RegistrationNumber
        AND NOT (NEW.EndDate <= StartDate OR NEW.StartDate >= EndDate)
    );
END;
""")

In [ ]:
# Section B: Create 5 Tuples for each relation in database
# Outlet
cursor.execute("""
INSERT INTO Outlet (OutletID, Address, PhoneNumber, FaxNumber) VALUES
(1, '123 Main St, Miami FL', 3051234567, 3051234568),
(2, '456 Oak Ave, Orlando FL', 4071234567, 4071234568),
(3, '789 Pine Rd, Tampa FL', 8131234567, 8131234568),
(4, '321 Elm St, Jacksonville FL', 9041234567, 9041234568),
(5, '654 Maple Dr, Tallahassee FL', 8501234567, 8501234568)
""")

# Vehicle
cursor.execute("""
INSERT INTO Vehicle (RegistrationNumber, OutletID, Model, Make, EngineSize, Capacity, CurrentMileage, DailyHireRate, CurrentLocation) VALUES
('ABC123', 1, 'Camry', 'Toyota', 2.5, 5.0, 15000, 49.99, 'Miami FL'),
('XYZ789', 2, 'Civic', 'Honda', 1.8, 5.0, 22000, 39.99, 'Orlando FL'),
('DEF456', 3, 'Mustang', 'Ford', 5.0, 4.0, 8000, 89.99, 'Tampa FL'),
('GHI101', 4, 'Altima', 'Nissan', 2.0, 5.0, 31000, 44.99, 'Jacksonville FL'),
('JKL202', 5, 'Malibu', 'Chevrolet', 1.5, 5.0, 12000, 34.99, 'Tallahassee FL')
""")

# Client
cursor.execute("""
INSERT INTO Client (ClientID, FirstName, LastName, Address, PhoneNumber, DateOfBirth, DrivingLicenseNumber) VALUES
(1, 'John', 'Smith', '10 River Rd, Miami FL', 3059876543, '1990-03-15', 'DL123456'),
(2, 'Maria', 'Garcia', '20 Lake Ave, Orlando FL', 4079876543, '1985-07-22', 'DL789012'),
(3, 'James', 'Brown', '30 Hill St, Tampa FL', 8139876543, '1992-11-05', 'DL345678'),
(4, 'Ashley', 'Jones', '40 Bay Dr, Jacksonville FL', 9049876543, '1998-01-30', 'DL901234'),
(5, 'Carlos', 'Rivera', '50 Coast Blvd, Tallahassee FL', 8509876543, '2000-06-18', 'DL567890')
""")

# Staff
cursor.execute("""
INSERT INTO Staff (StaffID, OutletID, FirstName, LastName, Address, PhoneNumber, DateOfBirth, Sex, DateJoined, JobTitle, Salary) VALUES
(1, 1, 'Linda', 'White', '11 Palm St, Miami FL', 3051112222, '1980-04-10', 'F', '2015-06-01', 'Manager', 55000.00),
(2, 2, 'Robert', 'Lee', '22 Oak Ave, Orlando FL', 4071112222, '1975-09-25', 'M', '2010-03-15', 'Supervisor', 48000.00),
(3, 3, 'Sarah', 'Kim', '33 Pine Rd, Tampa FL', 8131112222, '1990-12-01', 'F', '2018-08-20', 'Agent', 38000.00),
(4, 4, 'David', 'Chen', '44 Elm St, Jacksonville FL', 9041112222, '1988-02-14', 'M', '2017-01-10', 'Agent', 37000.00),
(5, 5, 'Emily', 'Davis', '55 Maple Dr, Tallahassee FL', 8501112222, '1995-07-07', 'F', '2020-05-05', 'Agent', 36000.00)
""")

# HireAgreement
cursor.execute("""
INSERT INTO HireAgreement (HireID, RegistrationNumber, ClientID, StartDate, EndDate, PreMileage, PostMileage) VALUES
(1, 'ABC123', 1, '2026-01-01', '2026-01-05', 15000, 15300),
(2, 'XYZ789', 2, '2026-01-10', '2026-01-15', 22000, 22400),
(3, 'DEF456', 3, '2026-02-01', '2026-02-03', 8000, 8150),
(4, 'GHI101', 4, '2026-02-20', '2026-02-25', 31000, 31500),
(5, 'JKL202', 5, '2026-03-01', '2026-03-07', 12000, 12600)
""")

conn.commit()

In [ ]:
# display Outlet relation
cursor.execute("""
SELECT * FROM Outlet;
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, '123 Main St, Miami FL', 3051234567, 3051234568)
(2, '456 Oak Ave, Orlando FL', 4071234567, 4071234568)
(3, '789 Pine Rd, Tampa FL', 8131234567, 8131234568)
(4, '321 Elm St, Jacksonville FL', 9041234567, 9041234568)
(5, '654 Maple Dr, Tallahassee FL', 8501234567, 8501234568)


In [ ]:
# display Vehicle relation
cursor.execute("""
SELECT * FROM Vehicle;
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

('ABC123', 1, 'Camry', 'Toyota', 2.5, 5.0, 15000, 49.99, 'Miami FL')
('XYZ789', 2, 'Civic', 'Honda', 1.8, 5.0, 22000, 39.99, 'Orlando FL')
('DEF456', 3, 'Mustang', 'Ford', 5.0, 4.0, 8000, 89.99, 'Tampa FL')
('GHI101', 4, 'Altima', 'Nissan', 2.0, 5.0, 31000, 44.99, 'Jacksonville FL')
('JKL202', 5, 'Malibu', 'Chevrolet', 1.5, 5.0, 12000, 34.99, 'Tallahassee FL')


In [ ]:
# display Client relation
cursor.execute("""
SELECT * FROM Client;
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'John', 'Smith', '10 River Rd, Miami FL', 3059876543, '1990-03-15', 'DL123456')
(2, 'Maria', 'Garcia', '20 Lake Ave, Orlando FL', 4079876543, '1985-07-22', 'DL789012')
(3, 'James', 'Brown', '30 Hill St, Tampa FL', 8139876543, '1992-11-05', 'DL345678')
(4, 'Ashley', 'Jones', '40 Bay Dr, Jacksonville FL', 9049876543, '1998-01-30', 'DL901234')
(5, 'Carlos', 'Rivera', '50 Coast Blvd, Tallahassee FL', 8509876543, '2000-06-18', 'DL567890')


In [ ]:
# display Staff relation
cursor.execute("""
SELECT * FROM Staff;
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 1, 'Linda', 'White', '11 Palm St, Miami FL', 3051112222, '1980-04-10', 'F', '2015-06-01', 'Manager', 55000.0)
(2, 2, 'Robert', 'Lee', '22 Oak Ave, Orlando FL', 4071112222, '1975-09-25', 'M', '2010-03-15', 'Supervisor', 48000.0)
(3, 3, 'Sarah', 'Kim', '33 Pine Rd, Tampa FL', 8131112222, '1990-12-01', 'F', '2018-08-20', 'Agent', 38000.0)
(4, 4, 'David', 'Chen', '44 Elm St, Jacksonville FL', 9041112222, '1988-02-14', 'M', '2017-01-10', 'Agent', 37000.0)
(5, 5, 'Emily', 'Davis', '55 Maple Dr, Tallahassee FL', 8501112222, '1995-07-07', 'F', '2020-05-05', 'Agent', 36000.0)


In [ ]:
# display HireAgreement relation
cursor.execute("""
SELECT * FROM HireAgreement;
""")
rows = cursor.fetchall()
for row in rows:
    print(row)

(1, 'ABC123', 1, '2026-01-01', '2026-01-05', 15000.0, 15300.0)
(2, 'XYZ789', 2, '2026-01-10', '2026-01-15', 22000.0, 22400.0)
(3, 'DEF456', 3, '2026-02-01', '2026-02-03', 8000.0, 8150.0)
(4, 'GHI101', 4, '2026-02-20', '2026-02-25', 31000.0, 31500.0)
(5, 'JKL202', 5, '2026-03-01', '2026-03-07', 12000.0, 12600.0)


In [ ]:
# Section C: SQL queries corresponding to Part 2.C
# Query 1
cursor.execute("""
SELECT
    S.StaffID,
    S.FirstName,
    S.LastName,
    S.JobTitle,
    S.Salary,
    O.OutletID,
    O.Address,
    O.PhoneNumber
FROM Staff S
JOIN Outlet O
    ON S.OutletID = O.OutletID;
""")

results = cursor.fetchall()
for row in results:
    print(row)

(1, 'Linda', 'White', 'Manager', 55000.0, 1, '123 Main St, Miami FL', 3051234567)
(2, 'Robert', 'Lee', 'Supervisor', 48000.0, 2, '456 Oak Ave, Orlando FL', 4071234567)
(3, 'Sarah', 'Kim', 'Agent', 38000.0, 3, '789 Pine Rd, Tampa FL', 8131234567)
(4, 'David', 'Chen', 'Agent', 37000.0, 4, '321 Elm St, Jacksonville FL', 9041234567)
(5, 'Emily', 'Davis', 'Agent', 36000.0, 5, '654 Maple Dr, Tallahassee FL', 8501234567)


In [ ]:
# Query 2
cursor.execute("""
SELECT
    C.ClientID,
    C.FirstName,
    C.LastName,
    C.PhoneNumber,
    HA.HireID,
    V.RegistrationNumber,
    V.Make,
    V.Model
FROM Client C
JOIN HireAgreement HA
    ON C.ClientID = HA.ClientID
JOIN Vehicle V
    ON HA.RegistrationNumber = V.RegistrationNumber
WHERE C.ClientID IN (
    SELECT ClientID
    FROM HireAgreement
    GROUP BY ClientID
    HAVING COUNT(*) > 1
);
""")

results = cursor.fetchall()
for row in results:
    print(row)

In [ ]:
# Query 3
cursor.execute("""
SELECT
    V.RegistrationNumber,
    V.Make,
    V.Model,
    V.EngineSize,
    V.Capacity,
    V.CurrentMileage,
    V.DailyHireRate,
    COUNT(HA.HireID) AS TotalHires
FROM Vehicle V
JOIN HireAgreement HA
    ON V.RegistrationNumber = HA.RegistrationNumber
GROUP BY
    V.RegistrationNumber,
    V.Make,
    V.Model,
    V.EngineSize,
    V.Capacity,
    V.CurrentMileage,
    V.DailyHireRate
ORDER BY TotalHires DESC
LIMIT 1;
""")

results = cursor.fetchall()
for row in results:
    print(row)

('ABC123', 'Toyota', 'Camry', 2.5, 5.0, 15000, 49.99, 1)


In [ ]:
# Query 4
cursor.execute("""
SELECT
    V.RegistrationNumber,
    V.Make,
    V.Model,
    C.ClientID,
    C.FirstName,
    C.LastName,
    HA.StartDate,
    HA.EndDate
FROM Vehicle V
JOIN HireAgreement HA
    ON V.RegistrationNumber = HA.RegistrationNumber
JOIN Client C
    ON HA.ClientID = C.ClientID
WHERE V.RegistrationNumber = 'GHI101';
""")

results = cursor.fetchall()
for row in results:
    print(row)

('GHI101', 'Nissan', 'Altima', 4, 'Ashley', 'Jones', '2026-02-20', '2026-02-25')


In [ ]:
# Query 5
cursor.execute("""
SELECT
    O.OutletID,
    O.Address,
    O.PhoneNumber,
    V.RegistrationNumber,
    V.Make,
    V.Model
FROM Outlet O
JOIN Vehicle V
    ON O.OutletID = V.OutletID
ORDER BY O.OutletID;
""")

results = cursor.fetchall()
for row in results:
    print(row)

(1, '123 Main St, Miami FL', 3051234567, 'ABC123', 'Toyota', 'Camry')
(2, '456 Oak Ave, Orlando FL', 4071234567, 'XYZ789', 'Honda', 'Civic')
(3, '789 Pine Rd, Tampa FL', 8131234567, 'DEF456', 'Ford', 'Mustang')
(4, '321 Elm St, Jacksonville FL', 9041234567, 'GHI101', 'Nissan', 'Altima')
(5, '654 Maple Dr, Tallahassee FL', 8501234567, 'JKL202', 'Chevrolet', 'Malibu')
